# 03 - Table 1 Generation (Blinded)

Last updated: A.L. 2026-04-13

This notebook reproduces the subset of manuscript Table 1 that is derivable from `data/BGA_merged_all_20260208_cleaned_for_analysis_blinded.csv` alone.

Included rows:
- Age
- Female (%)
- BIS total
- CPT measures
- Chalder total
- HADS Anxiety and Depression

Intentionally excluded because they are not available in the blinded dataset:
- Education
- RBANS index-score rows

Cfr. `02_table_1_generation.ipynb` (using `BGA_merged_all_20260208_cleaned.csv` - not in public repo)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

BIS_ITEMS = [f"BIS_Q{i}_BL" for i in range(1, 7)]
MEASURE_SPECS = [
    ("Age (years)", "TestAge"),
    ("BIS total", "BIS_total"),
    ("Detectability", "CPT_Detectability"),
    ("Omissions", "CPT_Omissions"),
    ("Commissions", "CPT_Commissions"),
    ("HRT", "CPT_HRT"),
    ("Chalder total", "Chalder_total"),
    ("Anxiety", "HADS_Anxiety"),
    ("Depression", "HADS_Depression"),
]


def resolve_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "BGA_merged_all_20260208_cleaned_for_analysis_blinded.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing the blinded CSV.")


ROOT = resolve_repo_root()
DATA_PATH = ROOT / "data" / "BGA_merged_all_20260208_cleaned_for_analysis_blinded.csv"

df = pd.read_csv(DATA_PATH, sep=";")
df.columns = df.columns.str.strip()

for col in ["Subject", "Gender", "Group", "IBStype"]:
    df[col] = df[col].astype(str).str.strip()

numeric_columns = [
    "TestAge",
    *BIS_ITEMS,
    "CPT_Detectability",
    "CPT_Omissions",
    "CPT_Commissions",
    "CPT_HRT",
    "TFS_Chalder",
    "HADS_Anxiety",
    "HADS_Depression",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["BIS_total"] = df[BIS_ITEMS].sum(axis=1, min_count=1)
df["Chalder_total"] = df["TFS_Chalder"]

ibs = df[df["Group"] == "IBS"].copy()
hc = df[df["Group"] == "HC"].copy()

print(f"Repo root: {ROOT}")
print(f"Data path:  {DATA_PATH}")
print(f"IBS n = {len(ibs)}, HC n = {len(hc)}")


In [ ]:
def female_mask(series: pd.Series) -> pd.Series:
    return series.astype(str).str.upper().str.startswith("F")


def cohens_d_pooled(x: pd.Series, y: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()
    nx, ny = len(x), len(y)
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    sp = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    return float((x.mean() - y.mean()) / sp)


def summarize_measure(ibs_df: pd.DataFrame, hc_df: pd.DataFrame, label: str, col: str) -> dict:
    ibs_vals = pd.to_numeric(ibs_df[col], errors="coerce").dropna()
    hc_vals = pd.to_numeric(hc_df[col], errors="coerce").dropna()
    test = stats.ttest_ind(ibs_vals, hc_vals, equal_var=False, nan_policy="omit")
    return {
        "Measure": label,
        "IBS_n": len(ibs_vals),
        "HC_n": len(hc_vals),
        "IBS_mean": ibs_vals.mean(),
        "IBS_sd": ibs_vals.std(ddof=1),
        "HC_mean": hc_vals.mean(),
        "HC_sd": hc_vals.std(ddof=1),
        "t": float(test.statistic),
        "p": float(test.pvalue),
        "d": cohens_d_pooled(ibs_vals, hc_vals),
    }


def p_stars(p: float) -> str:
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def fmt_mean_sd(mean: float, sd: float) -> str:
    return f"{mean:.1f} ({sd:.1f})"


def fmt_signed(value: float, digits: int = 2) -> str:
    rounded = round(float(value), digits)
    if rounded < 0:
        return f"${rounded:.{digits}f}$"
    return f"{rounded:.{digits}f}"


stats_df = pd.DataFrame([
    summarize_measure(ibs, hc, label, col) for label, col in MEASURE_SPECS
])

female_ibs = int(female_mask(ibs["Gender"]).sum())
female_hc = int(female_mask(hc["Gender"]).sum())
chi2, chi2_p, _, _ = stats.chi2_contingency([
    [female_ibs, len(ibs) - female_ibs],
    [female_hc, len(hc) - female_hc],
], correction=False)  # uncorrected Pearson chi-square (consistent with Cramer's V)

cohort_summary = {
    "IBS_n": len(ibs),
    "HC_n": len(hc),
    "female_pct_ibs": round(female_ibs / len(ibs) * 100, 1),
    "female_pct_hc": round(female_hc / len(hc) * 100, 1),
    "gender_chi2": float(chi2),
    "gender_p": float(chi2_p),
}

stats_df


,Measure,IBS_n,HC_n,IBS_mean,IBS_sd,HC_mean,HC_sd,t,p,d
0,Age (years),65,40,37.846154,11.373256,35.650000,12.517782,0.903577,3.690514e-01,0.185805
1,BIS total,58,39,17.603448,7.624843,10.307692,6.905962,4.890797,4.566798e-06,0.993196
2,Detectability,65,37,48.861538,7.837962,44.324324,7.063419,2.995946,3.622551e-03,0.599505
3,Omissions,65,37,47.353846,6.372379,45.027027,1.624235,2.789006,6.647122e-03,0.448309
4,Commissions,65,37,50.984615,9.075193,48.027027,8.050143,1.702311,9.245753e-02,0.339170
5,HRT,65,37,48.384615,7.997446,48.702703,8.768203,-0.181783,8.562827e-01,-0.038402
6,Chalder total,49,35,6.367347,3.425966,1.600000,2.499412,7.373551,1.202774e-10,1.549930
7,Anxiety,57,36,8.052632,4.210620,4.166667,3.281985,4.974461,3.268583e-06,1.001577
8,Depression,57,36,4.684211,3.094684,2.138889,2.319517,4.517446,1.931182e-05,0.902003


## Effect sizes with 95% confidence intervals

Reproduces the bracketed effect-size CIs reported in manuscript Table 1 (for the rows derivable from the blinded dataset):

- **Cohen's _d_** (pooled _SD_) with a large-sample **Hedges & Olkin** normal-approximation 95% CI.
- **Cramér's _V_** for the categorical sex comparison, from the **uncorrected Pearson** χ², with a **noncentral χ² (Smithson)** 95% CI.

The cell prints both a readable summary and the `[low, high]` strings used in the LaTeX table, so the CIs regenerate from code. (Education and the RBANS index rows are not in the blinded dataset; their CIs are produced by `02_table_1_generation.ipynb` on the full cohort file.)

In [ ]:
from scipy.optimize import brentq


def cohens_d_ci(x, y, conf=0.95):
    """Cohen's d (pooled SD) with a large-sample Hedges & Olkin 95% CI."""
    x = pd.to_numeric(x, errors="coerce").dropna()
    y = pd.to_numeric(y, errors="coerce").dropna()
    n1, n2 = len(x), len(y)
    sp = np.sqrt(((n1 - 1) * x.std(ddof=1) ** 2 + (n2 - 1) * y.std(ddof=1) ** 2) / (n1 + n2 - 2))
    d = (x.mean() - y.mean()) / sp
    se = np.sqrt((n1 + n2) / (n1 * n2) + d ** 2 / (2 * (n1 + n2)))
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    return float(d), float(d - z * se), float(d + z * se)


def cramers_v_ci(table, conf=0.95):
    """Cramer's V from the uncorrected Pearson chi-square, with a
    noncentral chi-square (Smithson) confidence interval."""
    table = np.asarray(table, dtype=float)
    chi2_obs = stats.chi2_contingency(table, correction=False)[0]
    n = table.sum()
    dof = (table.shape[0] - 1) * (table.shape[1] - 1)
    k = min(table.shape) - 1
    V = np.sqrt(chi2_obs / (n * k))
    alpha = 1 - conf

    def ncp(q):
        if stats.ncx2.cdf(chi2_obs, dof, 0.0) < q:
            return 0.0
        hi = 1.0
        while stats.ncx2.cdf(chi2_obs, dof, hi) > q:
            hi *= 2
        return brentq(lambda lam: stats.ncx2.cdf(chi2_obs, dof, lam) - q, 0.0, hi)

    lam_lo, lam_hi = ncp(1 - alpha / 2), ncp(alpha / 2)
    return float(V), float(np.sqrt(lam_lo / (n * k))), float(np.sqrt(lam_hi / (n * k))), float(chi2_obs)


print("Effect size [95% CI] -- reproduces manuscript Table 1\n")
for label, col in MEASURE_SPECS:
    d, lo, hi = cohens_d_ci(ibs[col], hc[col])
    print(f"{label:18s} d = {d:+.2f} [{lo:+.2f}, {hi:+.2f}]")

sex_table = [
    [female_ibs, len(ibs) - female_ibs],
    [female_hc, len(hc) - female_hc],
]
V, v_lo, v_hi, chi2_unc = cramers_v_ci(sex_table)
print(
    f"\nFemale (sex): uncorrected Pearson chi2 = {chi2_unc:.2f}, "
    f"Cramer's V = {V:.2f} [{v_lo:.2f}, {v_hi:.2f}]"
)

Effect size [95% CI] -- reproduces manuscript Table 1

Age (years)        d = +0.19 [-0.21, +0.58]
BIS total          d = +0.99 [+0.56, +1.42]
Detectability      d = +0.60 [+0.19, +1.01]
Omissions          d = +0.45 [+0.04, +0.86]
Commissions        d = +0.34 [-0.07, +0.75]
HRT                d = -0.04 [-0.44, +0.37]
Chalder total      d = +1.55 [+1.06, +2.04]
Anxiety            d = +1.00 [+0.56, +1.44]
Depression         d = +0.90 [+0.47, +1.34]

Female (sex): uncorrected Pearson chi2 = 1.56, Cramer's V = 0.12 [0.00, 0.31]


In [ ]:
def row_lookup(label: str) -> pd.Series:
    return stats_df.loc[stats_df["Measure"] == label].iloc[0]


summary_rows = [
    {
        "Measure": "Age (years)",
        "IBS": fmt_mean_sd(row_lookup("Age (years)")["IBS_mean"], row_lookup("Age (years)")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("Age (years)")["HC_mean"], row_lookup("Age (years)")["HC_sd"]),
        "t / chi^2": fmt_signed(row_lookup("Age (years)")["t"]),
        "Cohen's d": fmt_signed(row_lookup("Age (years)")["d"]),
    },
    {
        "Measure": "Female (%)",
        "IBS": f"{cohort_summary['female_pct_ibs']:.1f}",
        "HC": f"{cohort_summary['female_pct_hc']:.1f}",
        "t / chi^2": f"{cohort_summary['gender_chi2']:.2f}",
        "Cohen's d": "--",
    },
    {
        "Measure": "BIS total (0-42)",
        "IBS": fmt_mean_sd(row_lookup("BIS total")["IBS_mean"], row_lookup("BIS total")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("BIS total")["HC_mean"], row_lookup("BIS total")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('BIS total')['t'])}{p_stars(row_lookup('BIS total')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("BIS total")["d"]),
    },
    {
        "Measure": "Detectability (d')",
        "IBS": fmt_mean_sd(row_lookup("Detectability")["IBS_mean"], row_lookup("Detectability")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("Detectability")["HC_mean"], row_lookup("Detectability")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('Detectability')['t'])}{p_stars(row_lookup('Detectability')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("Detectability")["d"]),
    },
    {
        "Measure": "Omissions",
        "IBS": fmt_mean_sd(row_lookup("Omissions")["IBS_mean"], row_lookup("Omissions")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("Omissions")["HC_mean"], row_lookup("Omissions")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('Omissions')['t'])}{p_stars(row_lookup('Omissions')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("Omissions")["d"]),
    },
    {
        "Measure": "Commissions",
        "IBS": fmt_mean_sd(row_lookup("Commissions")["IBS_mean"], row_lookup("Commissions")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("Commissions")["HC_mean"], row_lookup("Commissions")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('Commissions')['t'])}{p_stars(row_lookup('Commissions')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("Commissions")["d"]),
    },
    {
        "Measure": "HRT",
        "IBS": fmt_mean_sd(row_lookup("HRT")["IBS_mean"], row_lookup("HRT")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("HRT")["HC_mean"], row_lookup("HRT")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('HRT')['t'])}{p_stars(row_lookup('HRT')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("HRT")["d"]),
    },
    {
        "Measure": "Chalder total (0-11)",
        "IBS": fmt_mean_sd(row_lookup("Chalder total")["IBS_mean"], row_lookup("Chalder total")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("Chalder total")["HC_mean"], row_lookup("Chalder total")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('Chalder total')['t'])}{p_stars(row_lookup('Chalder total')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("Chalder total")["d"]),
    },
    {
        "Measure": "Anxiety (0-21)",
        "IBS": fmt_mean_sd(row_lookup("Anxiety")["IBS_mean"], row_lookup("Anxiety")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("Anxiety")["HC_mean"], row_lookup("Anxiety")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('Anxiety')['t'])}{p_stars(row_lookup('Anxiety')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("Anxiety")["d"]),
    },
    {
        "Measure": "Depression (0-21)",
        "IBS": fmt_mean_sd(row_lookup("Depression")["IBS_mean"], row_lookup("Depression")["IBS_sd"]),
        "HC": fmt_mean_sd(row_lookup("Depression")["HC_mean"], row_lookup("Depression")["HC_sd"]),
        "t / chi^2": f"{fmt_signed(row_lookup('Depression')['t'])}{p_stars(row_lookup('Depression')['p'])}",
        "Cohen's d": fmt_signed(row_lookup("Depression")["d"]),
    },
]

summary_table = pd.DataFrame(summary_rows)
display(summary_table)

display(Markdown(
    "**Omitted from blinded-only reproduction:** Education and RBANS index-score rows."
))


,Measure,IBS,HC,t / chi^2,Cohen's d
0,Age (years),37.8 (11.4),35.6 (12.5),0.90,0.19
1,Female (%),78.5,67.5,1.56,--
2,BIS total (0-42),17.6 (7.6),10.3 (6.9),4.89***,0.99
3,Detectability (d'),48.9 (7.8),44.3 (7.1),3.00**,0.60
4,Omissions,47.4 (6.4),45.0 (1.6),2.79**,0.45
5,Commissions,51.0 (9.1),48.0 (8.1),1.70,0.34
6,HRT,48.4 (8.0),48.7 (8.8),$-0.18$,$-0.04$
7,Chalder total (0-11),6.4 (3.4),1.6 (2.5),7.37***,1.55
8,Anxiety (0-21),8.1 (4.2),4.2 (3.3),4.97***,1.00
9,Depression (0-21),4.7 (3.1),2.1 (2.3),4.52***,0.90


**Omitted from blinded-only reproduction:** Education and RBANS index-score rows.

In [ ]:
latex_lines = [
    r"\\begin{table}[htbp]",
    r"\\centering",
    r"\\caption{Blinded-data-supported subset of manuscript Table 1.}",
    r"\\label{tab:demographics_blinded_subset}",
    r"\\small",
    rf"\\begin{{tabular}}{{lcccc}}",
    r"\\toprule",
    rf"Measure & IBS ($n = {cohort_summary['IBS_n']}$) & HC ($n = {cohort_summary['HC_n']}$) & $t$ / $\\chi^2$ & Cohen's $d$ \\",
    r" & $M$ ($SD$) & $M$ ($SD$) & & \\",
    r"\\midrule",
]

for _, row in summary_table.iterrows():
    effect_size = row["Cohen's d"]
    latex_lines.append(
        f"{row['Measure']} & {row['IBS']} & {row['HC']} & {row['t / chi^2']} & {effect_size} \\\\"
    )

latex_lines.extend([
    r"\\bottomrule",
    r"\\end{tabular}",
    r"\\smallskip",
    r"\\footnotesize Education and RBANS index-score rows are omitted because they are not present in the blinded public dataset.",
    r"\\end{table}",
])

latex_table = "\n".join(latex_lines)
print(latex_table)


\\begin{table}[htbp]
\\centering
\\caption{Blinded-data-supported subset of manuscript Table 1.}
\\label{tab:demographics_blinded_subset}
\\small
\\begin{tabular}{lcccc}
\\toprule
Measure & IBS ($n = 65$) & HC ($n = 40$) & $t$ / $\\chi^2$ & Cohen's $d$ \\
 & $M$ ($SD$) & $M$ ($SD$) & & \\
\\midrule
Age (years) & 37.8 (11.4) & 35.6 (12.5) & 0.90 & 0.19 \\
Female (%) & 78.5 & 67.5 & 1.56 & -- \\
BIS total (0-42) & 17.6 (7.6) & 10.3 (6.9) & 4.89*** & 0.99 \\
Detectability (d') & 48.9 (7.8) & 44.3 (7.1) & 3.00** & 0.60 \\
Omissions & 47.4 (6.4) & 45.0 (1.6) & 2.79** & 0.45 \\
Commissions & 51.0 (9.1) & 48.0 (8.1) & 1.70 & 0.34 \\
HRT & 48.4 (8.0) & 48.7 (8.8) & $-0.18$ & $-0.04$ \\
Chalder total (0-11) & 6.4 (3.4) & 1.6 (2.5) & 7.37*** & 1.55 \\
Anxiety (0-21) & 8.1 (4.2) & 4.2 (3.3) & 4.97*** & 1.00 \\
Depression (0-21) & 4.7 (3.1) & 2.1 (2.3) & 4.52*** & 0.90 \\
\\bottomrule
\\end{tabular}
\\smallskip
\\footnotesize Education and RBANS index-score rows are omitted because they are not pr